# Chapter-16: Middlewares

# <font color="#800000">Middlewares</font>

## <font color="#3366FF">Node-Style Hooks</font>

### Basic Example Of Node-Style Hooks Middleware

#### Setup

In [1]:
import os
import random
import uuid
from typing import Any
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain.messages import HumanMessage,AIMessage
from langchain.agents import create_agent, AgentState
from langgraph.checkpoint.memory import InMemorySaver
from langchain.tools import tool
from langgraph.runtime import Runtime
from langchain.agents.middleware import before_agent, after_agent,before_model, after_model

## Loading OpenAI API key from env file into an environment variable
load_dotenv()
os.environ["OPENAI_API_KEY"] = os.getenv('OPENAI_API_KEY')

## Initialize ChatOpenAI LLM
llm = ChatOpenAI(model="gpt-5-mini")

## Tool
@tool
def stock_price(name: str) -> int:
    """ Fetches the current market price of a stock."""
    price = round(random.uniform(10, 500), 2)
    return price

## -------------- Middlewares ----------------
@before_agent
def log_before_agent(state: AgentState, runtime: Runtime):
    print(f"before_agent: Starting agent with {len(state['messages'])} messages")
    return None

@after_agent
def log_after_agent(state: AgentState, runtime: Runtime):
    print(f"after_agent: Agent completed with {len(state['messages'])} messages")
    return None

@before_model
def log_before_model(state: AgentState, runtime: Runtime):
    print(f"before_model: About to call model with {len(state['messages'])} messages")
    return None

@after_model
def log_after_model(state: AgentState, runtime: Runtime):
    
    if (state["messages"][-1].content != ''):
        print("after_model:", state["messages"][-1].content)
    else:
        print("after_model:", state["messages"][-1].tool_calls)        
    return None


## Define Agent
agent = create_agent(
    model=llm,
    tools=[stock_price],
    system_prompt="You are a stock market assistant. Respond in one line",
    middleware=[log_before_agent,log_after_agent,log_before_model,log_after_model],
    checkpointer=InMemorySaver(),  
)


#### Invoke Agent

In [2]:
## Create config with thread ID
config = {"configurable": {"thread_id": uuid.uuid4()}}

## Invoke agent
response = agent.invoke({"messages": [HumanMessage(content="What is stock price of Tesla?")]},config)

## Print Messages
for m in response['messages']:
    m.pretty_print()


before_agent: Starting agent with 1 messages
before_model: About to call model with 1 messages
after_model: [{'name': 'stock_price', 'args': {'name': 'TSLA'}, 'id': 'call_7knoNMIoRsbPoWtGv46uLuQx', 'type': 'tool_call'}]
before_model: About to call model with 3 messages
after_model: Tesla (TSLA) current price: $474.72 USD.
after_agent: Agent completed with 4 messages
================================ Human Message =================================

What is stock price of Tesla?
================================== Ai Message ==================================
Tool Calls:
  stock_price (call_7knoNMIoRsbPoWtGv46uLuQx)
 Call ID: call_7knoNMIoRsbPoWtGv46uLuQx
  Args:
    name: TSLA
================================= Tool Message =================================
Name: stock_price

474.72
================================== Ai Message ==================================

Tesla (TSLA) current price: $474.72 USD.


### Jump To Nodes In Hook-Style Middlewares

#### Example

In [3]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain.messages import HumanMessage,AIMessage
from langchain.agents import create_agent, AgentState
from langgraph.runtime import Runtime
from langchain.agents.middleware import before_agent

## Loading OpenAI API key from env file into an environment variable
load_dotenv()
os.environ["OPENAI_API_KEY"] = os.getenv('OPENAI_API_KEY')

## Initialize ChatOpenAI LLM
llm = ChatOpenAI(model="gpt-5-mini")

## Middleware
@before_agent(can_jump_to=["end"])
def agent_guardrail(state: AgentState, runtime: Runtime):
    if 'stupid' in state['messages'][-1].content:
        print('BLOCKING REQUEST TO AGENT: Please do not include inappropriate words in the input!')
        return {"jump_to": "end"}
    else:
        return None

## Define Agent
agent = create_agent(
    model=llm,
    system_prompt="You are a helpful assistant. Respond in one line",
    middleware=[agent_guardrail]
)

## Invoke agent
response = agent.invoke({"messages": [HumanMessage(content="You are stupid")]})

## Print Messages
for m in response['messages']:
    m.pretty_print()


BLOCKING REQUEST TO AGENT: Please do not include inappropriate words in the input!
================================ Human Message =================================

You are stupid


### Use Of Custom States In Hook-Style Middlewares

#### Example

In [4]:
import os
import uuid
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from typing_extensions import NotRequired
from langchain.tools import tool
from langchain.messages import HumanMessage,AIMessage
from langchain.agents import create_agent, AgentState
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.runtime import Runtime
from langchain.agents.middleware import before_agent, after_agent

## Loading OpenAI API key from env file into an environment variable
load_dotenv()
os.environ["OPENAI_API_KEY"] = os.getenv('OPENAI_API_KEY')

## Initialize ChatOpenAI LLM
llm = ChatOpenAI(model="gpt-5-mini")

## CustomState
class CustomState(AgentState):
    agent_call_counter: NotRequired[int]    

## Middleware to check if the agent call limit is reached
@before_agent(state_schema=CustomState, can_jump_to=["end"])
def check_call_limit(state: CustomState, runtime: Runtime):
    count = state.get("agent_call_counter", 1)
    if count > 3:
        print(f"Agent call {count}: Agent call limit reached. Please try again later!")
        return {"jump_to": "end"}
    print(f"Agent call {count}: {state['messages'][-1].content}")
    return None

## Middleware to increment counter after each agent call
@after_agent(state_schema=CustomState)
def increment_counter(state: CustomState, runtime: Runtime):
    return {"agent_call_counter": state.get("agent_call_counter", 1) + 1}

## Define Agent
agent = create_agent(
    model=llm,
    system_prompt="You are a helpful assistant. Respond in one line",
    middleware=[check_call_limit,increment_counter],
    checkpointer=InMemorySaver()
)

## Create config with thread ID
config = {"configurable": {"thread_id": uuid.uuid4()}}

## Invoke agent (1st Time)
response = agent.invoke({"messages": [HumanMessage(content="What is 2+2")]},config)
## Invoke agent (2nd Time)
response = agent.invoke({"messages": [HumanMessage(content="Add 5 to it")]},config)
## Invoke agent (3rd Time)
response = agent.invoke({"messages": [HumanMessage(content="Multiply it by 10")]},config)
## Invoke agent (4th Time) ### Request will get block ####
response = agent.invoke({"messages": [HumanMessage(content="Subtract 100 from it")]},config)


Agent call 1: What is 2+2
Agent call 2: Add 5 to it
Agent call 3: Multiply it by 10
Agent call 4: Agent call limit reached. Please try again later!


### Using Multiple Middlewares At Same Hook

#### Example

In [5]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain.messages import HumanMessage,AIMessage
from langchain.agents import create_agent, AgentState
from langgraph.runtime import Runtime
from langchain.agents.middleware import before_model, after_model

## Loading OpenAI API key from env file into an environment variable
load_dotenv()
os.environ["OPENAI_API_KEY"] = os.getenv('OPENAI_API_KEY')

## Initialize ChatOpenAI LLM
llm = ChatOpenAI(model="gpt-5-mini")

## -------------- Middlewares ----------------
@before_model
def log_before_model_1(state: AgentState, runtime: Runtime):
    print(f"before_model: We are using OpenAI model.")
    return None

@before_model
def log_before_model_2(state: AgentState, runtime: Runtime):
    print(f"before_model: About to call model.")
    return None    

@after_model
def log_after_model_1(state: AgentState, runtime: Runtime):
    print(f"after_model: Completed model call.")
    return None

@after_model
def log_after_model_2(state: AgentState, runtime: Runtime):
    print(f"after_model: It was nice serving you!")
    return None

## Define Agent
agent = create_agent(
    model=llm,
    system_prompt="You are a helpful assistant. Respond in one line",
    middleware=[log_before_model_1,log_before_model_2, log_after_model_1, log_after_model_2]
)

## Invoke agent
response = agent.invoke({"messages": [HumanMessage(content="What is 2+2")]})

## Print Messages
for m in response['messages']:
    m.pretty_print()


before_model: We are using OpenAI model.
before_model: About to call model.
after_model: It was nice serving you!
after_model: Completed model call.
================================ Human Message =================================

What is 2+2
================================== Ai Message ==================================

2 + 2 = 4.


## <font color="#3366FF">Wrap-Style Hooks</font>

### Understanding wrap_model_call Hook

#### Example

In [6]:
import os
from dotenv import load_dotenv
from typing import Callable
from langchain_openai import ChatOpenAI
from langchain.messages import HumanMessage,AIMessage
from langchain.agents import create_agent
from langchain.agents.middleware import wrap_model_call, ModelResponse, ModelRequest

## Loading OpenAI API key from env file into an environment variable
load_dotenv()
os.environ["OPENAI_API_KEY"] = os.getenv('OPENAI_API_KEY')

## Initialize ChatOpenAI LLM
llm = ChatOpenAI(model="gpt-5-mini")

## Middleware
@wrap_model_call
def change_model_call(request: ModelRequest, handler: Callable[[ModelRequest], ModelResponse]):

    ## Override System Message(Prompt) to return joke instead of fact
    request = request.override(system_message="You always return a one-line joke on the topic provided by the user")

    ## Call Model (Returns ModelResponse object)
    response = handler(request)

    ## Extract AIMessage from ModelResponse
    ai_msg = response.result[0]

    ## Return ModelResponse with AIMessage content in uppercase
    return ModelResponse(result=[AIMessage(content=ai_msg.content.upper())])

## Define Agent
agent = create_agent(
    model=llm,
    system_prompt="You always return a one-line fact on the topic provided by the user",
    middleware=[change_model_call]
)

## Invoke agent
response = agent.invoke({"messages": [HumanMessage(content="Cats")]})

## Print Messages
for m in response['messages']:
    m.pretty_print()


================================ Human Message =================================

Cats
================================== Ai Message ==================================

WHY DID THE CAT SIT ON THE COMPUTER? TO KEEP AN EYE ON THE MOUSE.


### Understanding wrap_tool_call Hook

#### Example

In [7]:
import os
import random
from dotenv import load_dotenv
from typing import Callable
from langchain_openai import ChatOpenAI
from langchain.messages import HumanMessage
from langchain.tools import tool
from langchain.agents import create_agent
from langchain.agents.middleware import wrap_tool_call
from langchain.tools.tool_node import ToolCallRequest
from langchain.messages import ToolMessage


## Loading OpenAI API key from env file into an environment variable
load_dotenv()
os.environ["OPENAI_API_KEY"] = os.getenv('OPENAI_API_KEY')

## Initialize ChatOpenAI LLM
llm = ChatOpenAI(model="gpt-5-mini")

## Tool
@tool
def stock_price(name: str) -> int:
    """ Fetches the current market price of a stock."""
    price = round(random.uniform(10, 500), 2)
    return price

## Middleware to convert 'name' argument of Tool into uppercase
@wrap_tool_call
def uppercase_args(request: ToolCallRequest, handler: Callable[[ToolCallRequest], ToolMessage]):
    print(f"Executing tool: {request.tool_call['name']}")
    print(f"Arguments: {request.tool_call['args']}")
    
    ## Converting 'name' arguments to uppercase
    request.tool_call['args']['name'] = request.tool_call['args']['name'].upper()
    print(f"Modified Arguments: {request.tool_call['args']}")

    ## Executing Tool Call with modified request
    result = handler(request)

    return result


## Define Agent
agent = create_agent(
    model=llm,
    tools=[stock_price],
    middleware=[uppercase_args],
    system_prompt="You are a helpful stock market assistant",
)

## Invoke agent
response = agent.invoke({"messages": [HumanMessage(content="What is stock price of Google")]})

## Print Messages
for m in response['messages']:
    m.pretty_print()


Executing tool: stock_price
Arguments: {'name': 'Google'}
Modified Arguments: {'name': 'GOOGLE'}
================================ Human Message =================================

What is stock price of Google
================================== Ai Message ==================================
Tool Calls:
  stock_price (call_utJyv2mtXOB8mlWzAq3LszUQ)
 Call ID: call_utJyv2mtXOB8mlWzAq3LszUQ
  Args:
    name: GOOGLE
================================= Tool Message =================================
Name: stock_price

478.61
================================== Ai Message ==================================

I fetched the current price for Google (Alphabet) as $478.61 per share. 

Note: Alphabet trades as GOOGL (Class A) and GOOG (Class C); prices can differ slightly and may be delayed depending on the source. Do you want the quote for a specific ticker (GOOGL or GOOG), real-time updates, or historical data?


## <font color="#3366FF">Built-In Middlewares</font>

### Model Call Limit Middleware

#### Example

In [8]:
import os
import uuid
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain.messages import HumanMessage,AIMessage
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents.middleware import ModelCallLimitMiddleware

## Loading OpenAI API key from env file into an environment variable
load_dotenv()
os.environ["OPENAI_API_KEY"] = os.getenv('OPENAI_API_KEY')

## Initialize ChatOpenAI LLM
llm = ChatOpenAI(model="gpt-5-mini")

## Define Agent
agent = create_agent(
    model=llm,
    system_prompt="You are a helpful assistant and reply in one line always.",
    middleware=[
        ModelCallLimitMiddleware(
            thread_limit=2, ## Limit across multiple invocations in a single thread
            run_limit=5, ## Limit during single invocation
            exit_behavior="end",
        ),
    ],
    checkpointer=InMemorySaver(),  
)

## Create config with thread ID
config = {"configurable": {"thread_id": uuid.uuid4()}}

## Invoke agent (1st Time)
response = agent.invoke({"messages": [HumanMessage(content="What is 2+2")]},config)
## Invoke agent (2nd Time)
response = agent.invoke({"messages": [HumanMessage(content="Add 5 to it")]},config)
## Invoke agent (3rd Time) ### Model Call limit will hit ####
response = agent.invoke({"messages": [HumanMessage(content="Multiply it by 5")]},config)

## Print Messages
for m in response['messages']:
    m.pretty_print()


================================ Human Message =================================

What is 2+2
================================== Ai Message ==================================

4
================================ Human Message =================================

Add 5 to it
================================== Ai Message ==================================

9
================================ Human Message =================================

Multiply it by 5
================================== Ai Message ==================================

Model call limits exceeded: thread limit (2/2)


### Tool Call Limit Middleware

#### Example

In [9]:
import os
import uuid
import random
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain.messages import HumanMessage,AIMessage
from langchain.tools import tool
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents.middleware import ToolCallLimitMiddleware

## Loading OpenAI API key from env file into an environment variable
load_dotenv()
os.environ["OPENAI_API_KEY"] = os.getenv('OPENAI_API_KEY')

## Initialize ChatOpenAI LLM
llm = ChatOpenAI(model="gpt-5-mini")

## Tool
@tool
def stock_price(name: str) -> int:
    """ Fetches the current market price of a stock."""
    price = round(random.uniform(10, 500), 2)
    return price

## Define Agent
agent = create_agent(
    model=llm,
    system_prompt="You are a helpful assistant and reply in one line always.",
    tools=[stock_price],
    middleware= [
        ToolCallLimitMiddleware(
        tool_name="stock_price",
        thread_limit=3, ## Limit across multiple invocations in a single thread
        run_limit=1, ## Limit during single invocation
        exit_behavior = 'continue'
    ),
    ],
    checkpointer=InMemorySaver(),  
)

## Create config with thread ID
config = {"configurable": {"thread_id": uuid.uuid4()}}

## Invoke agent
response = agent.invoke({"messages": [HumanMessage(content="What are stock prices of Microsoft and Intel?")]},config)

## Print Messages
for m in response['messages']:
    m.pretty_print()


================================ Human Message =================================

What are stock prices of Microsoft and Intel?
================================== Ai Message ==================================
Tool Calls:
  stock_price (call_SBJDrpzq0aSt8fsbpHlNNYsh)
 Call ID: call_SBJDrpzq0aSt8fsbpHlNNYsh
  Args:
    name: Microsoft
  stock_price (call_37wZEfPUxVpihp8GOPhe6lBx)
 Call ID: call_37wZEfPUxVpihp8GOPhe6lBx
  Args:
    name: Intel
================================= Tool Message =================================
Name: stock_price

Tool call limit exceeded. Do not call 'stock_price' again.
================================= Tool Message =================================
Name: stock_price

50.83
================================== Ai Message ==================================

Microsoft: $50.83; Intel: unable to fetch due to a "tool call limit exceeded" error — would you like me to try again, use an approximate/latest known value, or check a specific time?


### Model Fallback Middleware

#### Example

In [10]:
import os
import uuid
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain.messages import HumanMessage,AIMessage
from langchain.agents import create_agent
from langchain.agents.middleware import ModelFallbackMiddleware

## Loading OpenAI API key from env file into an environment variable
load_dotenv()
os.environ["OPENAI_API_KEY"] = os.getenv('OPENAI_API_KEY')

## Initialize ChatOpenAI LLM (non-existing model)
llm = ChatOpenAI(model="gpt-xyz")

## Define Agent
agent = create_agent(
    model=llm,
    system_prompt="You are a helpful assistant and reply in one line always.",
    middleware=[
        ModelFallbackMiddleware(
            "gpt-5-mini",
            "gpt-5",
        ),
    ]
)

## Invoke agent
response = agent.invoke({"messages": [HumanMessage(content="What is supernova?")]})

## Print Messages
for m in response['messages']:
    m.pretty_print()


================================ Human Message =================================

What is supernova?
================================== Ai Message ==================================

A supernova is a catastrophic stellar explosion that marks the end of a star’s life—caused by core collapse in massive stars or thermonuclear runaway in white dwarfs—releasing enormous energy, forging heavy elements, briefly outshining galaxies, and leaving behind a neutron star, black hole, or expanding remnant.


## <font color="#3366FF">Human-In-The-Loop Middleware</font>

### Example Of HumanInTheLoopMiddleware

#### Setup

In [11]:
import os
import random
import uuid
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain.messages import HumanMessage
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver
from langchain.tools import tool
from langgraph.types import Command
from langchain.agents.middleware import HumanInTheLoopMiddleware 


## Loading OpenAI API key from env file into an environment variable
load_dotenv()
os.environ["OPENAI_API_KEY"] = os.getenv('OPENAI_API_KEY')

## Initialize ChatOpenAI LLM
llm = ChatOpenAI(model="gpt-5-mini")

## Tool
@tool
def buy_stock(name: str, qty: int) -> str:
    """
    This function buys a specific quantity of a stock.
    """
    return f"Successfully purchased {qty} shares of {name}."

@tool
def sell_stock(name: str, qty: int) -> str:
    """
    This function sells a specific quantity of a stock.
    """
    return f"Successfully sold {qty} shares of {name}."

@tool
def stock_price(name: str) -> int:
    """ Fetches the current market price of a stock."""
    price = round(random.uniform(10, 500), 2)
    return price


## Define Agent
agent = create_agent(
    model=llm,
    tools=[buy_stock,sell_stock,stock_price],
    system_prompt="You are a stock market assistant. Respond in one line",
    middleware=[
        HumanInTheLoopMiddleware( 
            interrupt_on={
                "buy_stock": True,  # All decisions (approve, edit, reject) allowed
                "sell_stock": {"allowed_decisions": ["approve", "reject"], "description":"Approve sell of stocks"},  # No editing allowed
                "stock_price": False, # No approval required
            },
            description_prefix="Tool execution pending approval",
        ),
    ],
    checkpointer=InMemorySaver(),  
)


#### Scenario 1: Approve Tool Call

In [12]:
## Create config with thread ID
config = {"configurable": {"thread_id": uuid.uuid4()}}

## Invoke agent
response = agent.invoke({"messages": [HumanMessage(content="Buy 20 stocks of Tesla")]},config)

## Print Messages
for m in response['messages']:
    m.pretty_print()


================================ Human Message =================================

Buy 20 stocks of Tesla
================================== Ai Message ==================================
Tool Calls:
  buy_stock (call_YXuksn0dP8HdARHi6PrVw3tP)
 Call ID: call_YXuksn0dP8HdARHi6PrVw3tP
  Args:
    name: TSLA
    qty: 20


In [13]:
## Print Interrupt Details
print(response['__interrupt__'])


[Interrupt(value={'action_requests': [{'name': 'buy_stock', 'args': {'name': 'TSLA', 'qty': 20}, 'description': "Tool execution pending approval\n\nTool: buy_stock\nArgs: {'name': 'TSLA', 'qty': 20}"}], 'review_configs': [{'action_name': 'buy_stock', 'allowed_decisions': ['approve', 'edit', 'reject', 'respond']}]}, id='fb9899b9a598a60392284c0457ff677a')]


In [14]:
## Approve Tool Call
response = agent.invoke(Command( resume={"decisions": [{"type": "approve"}]}), config)

## Print Messages
for m in response['messages']:
    m.pretty_print()


================================ Human Message =================================

Buy 20 stocks of Tesla
================================== Ai Message ==================================
Tool Calls:
  buy_stock (call_YXuksn0dP8HdARHi6PrVw3tP)
 Call ID: call_YXuksn0dP8HdARHi6PrVw3tP
  Args:
    name: TSLA
    qty: 20
================================= Tool Message =================================
Name: buy_stock

Successfully purchased 20 shares of TSLA.
================================== Ai Message ==================================

Confirmed — purchased 20 shares of TSLA.


#### Scenario 2: Edit Tool Call

In [15]:
## Create config with thread ID
config = {"configurable": {"thread_id": uuid.uuid4()}}

## Invoke agent
response = agent.invoke({"messages": [HumanMessage(content="Buy 35 stocks of Nvidia")]},config)

## Print Interrupt Details
print(response['__interrupt__'])


[Interrupt(value={'action_requests': [{'name': 'buy_stock', 'args': {'name': 'Nvidia', 'qty': 35}, 'description': "Tool execution pending approval\n\nTool: buy_stock\nArgs: {'name': 'Nvidia', 'qty': 35}"}], 'review_configs': [{'action_name': 'buy_stock', 'allowed_decisions': ['approve', 'edit', 'reject', 'respond']}]}, id='5c1ec03c1409c0218f96b914feddd053')]


In [16]:
## Edit Tool Call
response = agent.invoke(Command( resume={"decisions": [
                {
                    "type": "edit",
                    "edited_action": {
                        "name": "buy_stock",
                        "args": {"name": "Nvidia", "qty": 50},
                    }
                }
            ]}), config)

## Print Messages
for m in response['messages']:
    m.pretty_print()


================================ Human Message =================================

Buy 35 stocks of Nvidia
================================== Ai Message ==================================
Tool Calls:
  buy_stock (call_ZumEtflHwmgEv9YAXCu6318k)
 Call ID: call_ZumEtflHwmgEv9YAXCu6318k
  Args:
    name: Nvidia
    qty: 50
================================= Tool Message =================================
Name: buy_stock

Successfully purchased 50 shares of Nvidia.
================================== Ai Message ==================================

I purchased 50 shares of Nvidia (you requested 35); would you like me to sell 15 to match your request?


#### Scenario 3: Reject Tool Call

In [17]:
## Create config with thread ID
config = {"configurable": {"thread_id": uuid.uuid4()}}

## Invoke agent
response = agent.invoke({"messages": [HumanMessage(content="Sell 20 stocks of Google")]},config)

## Print Interrupt Details
print(response['__interrupt__'])


[Interrupt(value={'action_requests': [{'name': 'sell_stock', 'args': {'name': 'Google', 'qty': 20}, 'description': 'Approve sell of stocks'}], 'review_configs': [{'action_name': 'sell_stock', 'allowed_decisions': ['approve', 'reject']}]}, id='7d76f00470e0aac42386ce165358b794')]


In [18]:
## Reject Tool Call
response = agent.invoke(Command( resume={"decisions": [{"type": "reject"}]}), config)

## Print Messages
for m in response['messages']:
    m.pretty_print()


================================ Human Message =================================

Sell 20 stocks of Google
================================== Ai Message ==================================
Tool Calls:
  sell_stock (call_Y0dh4qKr36fOm9zkau3qVU9f)
 Call ID: call_Y0dh4qKr36fOm9zkau3qVU9f
  Args:
    name: Google
    qty: 20
================================= Tool Message =================================
Name: sell_stock

User rejected the tool call for `sell_stock` with id call_Y0dh4qKr36fOm9zkau3qVU9f. The tool was not executed. Do not retry this tool call unless the user explicitly requests it.
================================== Ai Message ==================================

I didn't execute the sale because you rejected the tool call — please confirm you want to sell 20 shares of Google (GOOG or GOOGL?), and specify market or limit order (if limit, state the price).
